# Sylvester's AI Lab — Colab Studio v11
**One-click deploy: LTX-2.3 video + FLUX image + ReActor + RIFE + Voice Clone + Wav2Lip + edge-tts**

Runtime → Change runtime type → **T4 GPU** (or better).

Press **⌘+F9** (or Runtime → Run all) and wait ~15min for model downloads.

---

## 1. GPU Check

In [ ]:
import subprocess, os, sys, json, time, threading, urllib.request
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True,text=True).stdout.strip()
print(f'Detected GPU: {r}')
assert 'T4' in r or 'L4' in r or 'A100' in r or 'V100' in r or 'P100' in r, 'Need a GPU!'
os.chdir('/content')

## 2. Install System Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq git ffmpeg aria2 2>&1 | tail -2
# Install everything in one shot so pip resolves deps correctly
# Pin numpy first so insightface/onnx don't pull 2.x back in
!pip install numpy==1.26.4 onnxruntime insightface huggingface-hub edge-tts gradio pillow requests diffusers einops kornia "transformers[timm]" gdown 2>&1 | tail -5
# Coqui TTS (optional — wrap in try/except since it may not be on all PyPI mirrors)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'TTS'], capture_output=True, timeout=120)
# Monkey-patch numpy FPE flag (needed when numpy 1.x loads .so compiled against numpy 2.x)
import numpy as np
try:
    np._core._multiarray_umath._blas_supports_fpe = True
except AttributeError:
    try:
        np.core._multiarray_umath._blas_supports_fpe = True
    except AttributeError:
        pass
# Verify insightface
import insightface; print(f'insightface {insightface.__version__} OK')


## 3. Install ComfyUI + Custom Nodes

In [ ]:
BASE = '/content/studio'
COMFY = f'{BASE}/ComfyUI'
os.makedirs(BASE, exist_ok=True)

if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git "{COMFY}" 2>&1 | tail -2
    !pip install -q -r "{COMFY}/requirements.txt" 2>&1 | tail -2
    print('ComfyUI installed')
else:
    print('ComfyUI already exists')

# Custom nodes
NODES = f'{COMFY}/custom_nodes'
os.makedirs(NODES, exist_ok=True)
for name, url in [
    ('ComfyUI-VideoHelperSuite','https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'),
    ('ComfyUI_KJNodes','https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI-Manager','https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI-Frame-Interpolation','https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git'),
    ('ComfyUI-LTXVideo','https://github.com/Lightricks/ComfyUI-LTXVideo.git'),
    ('comfyui-reactor-node','https://codeberg.org/Gourieff/comfyui-reactor-node.git'),
]:
    p = f'{NODES}/{name}'
    if not os.path.isdir(p):
        !git clone --depth 1 "{url}" "{p}" 2>&1 | tail -1
    else:
        print(f'{name}: exists')

for req in ['comfyui-reactor-node/requirements.txt','ComfyUI-Frame-Interpolation/requirements.txt']:
    rp = f'{NODES}/{req}'
    if os.path.exists(rp):
        !pip install -q -r "{rp}" 2>&1 | tail -2

print('ComfyUI + nodes ready')

## 4. Download Model Weights (takes the longest)

In [ ]:
MODELS = f'{COMFY}/models'
for d in ['clip','clip_vision','vae','diffusion_models','loras','onnx','ultralytics']:
    os.makedirs(f'{MODELS}/{d}', exist_ok=True)

# Set your HF_TOKEN in Colab secrets (key: HF_TOKEN) or uncomment below
# HF_TOKEN = 'your_token_here'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN not set. Go to Tools → Secrets → add HF_TOKEN.')

def dl(url, path, label=''):
    if os.path.exists(path) and os.path.getsize(path) > 1e6:
        print(f'{label}: exists ({os.path.getsize(path)/1e6:.0f}MB)')
        return
    !curl -L -H "Authorization: Bearer {HF_TOKEN}" -o "{path}" "{url}" --progress-bar 2>&1 | tail -1
    sz = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'{label}: {sz/1e6:.0f}MB')

# CLIP / T5 / AE
dl('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
   f'{MODELS}/clip/clip_l.safetensors', 'CLIP-L')
dl('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
   f'{MODELS}/clip/t5xxl_fp8_e4m3fn.safetensors', 'T5XXL')
dl('https://huggingface.co/Kijai/flux-fp8/resolve/main/flux-vae-bf16.safetensors',
   f'{MODELS}/vae/ae.safetensors', 'AE')

# FLUX schnell fp8 (~17GB)
fp = f'{MODELS}/diffusion_models/flux1-dev-fp8-e4m3fn.safetensors'
if not os.path.exists(fp):
    !aria2c -x 4 -s 4 --header="Authorization: Bearer {HF_TOKEN}" \
      'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8-e4m3fn.safetensors' \
      -d "{MODELS}/diffusion_models" -o flux1-dev-fp8-e4m3fn.safetensors 2>&1 | tail -3
sz = os.path.getsize(fp)/1e9 if os.path.exists(fp) else 0
print(f'FLUX: {sz:.1f}GB')

# LTX distilled (~5GB, 2B param, fits T4)
lp = f'{MODELS}/diffusion_models/ltx-2.3-22b-distilled-1.1.safetensors'
if not os.path.exists(lp):
    !aria2c -x 4 -s 4 --header="Authorization: Bearer {HF_TOKEN}" \
      'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.safetensors' \
      -d "{MODELS}/diffusion_models" -o ltx-2.3-22b-distilled-1.1.safetensors 2>&1 | tail -3
sz = os.path.getsize(lp)/1e9 if os.path.exists(lp) else 0
print(f'LTX: {sz:.1f}GB')

# ReActor models
dl('https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/GFPGANv1.4.pth',
   f'{MODELS}/facerestore_models/GFPGANv1.4.pth', 'GFPGAN')
dl('https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx',
   f'{MODELS}/onnx/inswapper_128.onnx', 'inswapper')

# InsightFace
from insightface.model_zoo import get_model
get_model('buffalo_l', download=True, download_zip=True)
print('InsightFace: OK')
# Symlinks for ReActor model paths
INSIGHTFACE_DIR = f'{MODELS}/insightface'
os.makedirs(INSIGHTFACE_DIR, exist_ok=True)
# Symlink inswapper_128.onnx to where Reactor expects it
if not os.path.exists(f'{INSIGHTFACE_DIR}/inswapper_128.onnx') and os.path.exists(f'{MODELS}/onnx/inswapper_128.onnx'):
    os.symlink(f'{MODELS}/onnx/inswapper_128.onnx', f'{INSIGHTFACE_DIR}/inswapper_128.onnx')
# Symlink buffalo_l to Reactor node
REACTOR_NODE = f'{COMFY}/custom_nodes/comfyui-reactor-node'
os.makedirs(REACTOR_NODE, exist_ok=True)
if not os.path.exists(f'{REACTOR_NODE}/inswapper_128.onnx') and os.path.exists(f'{MODELS}/onnx/inswapper_128.onnx'):
    os.symlink(f'{MODELS}/onnx/inswapper_128.onnx', f'{REACTOR_NODE}/inswapper_128.onnx')

# Patch flux_dev.json CLIP name
flux_wf = f'{BASE}/flux_dev.json'
if os.path.exists(flux_wf):
    import json as _j
    with open(flux_wf) as _f:
        _wf = _j.load(_f)
    _wf['clip_name2']['inputs']['clip_name'] = 't5xxl_fp8_e4m3fn.safetensors'
    with open(flux_wf, 'w') as _f:
        _j.dump(_wf, _f, indent=2)
    print('flux_dev.json: CLIP name patched')

print('\n=== All weights ready ===')

## 5. Download Studio Bundle (Python modules + app)

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/fmssylvester/sylvesters-ai-lab/main'

FILES = [
    'launch_app.py', 'upscaler.py', 'assets_b64.py',
    'voiceover.py', 'voice_cloner.py', 'swapper.py',
    'interpolator.py', 'scene.py', 'avatars.py', 'director.py',
    'shotbuilder.html', 'ltx_api_workflow.json', 'flux_dev.json',
]

for fn in FILES:
    dst = f'{BASE}/{fn}'
    if os.path.exists(dst):
        print(f'{fn}: exists')
        continue
    url = f'{GITHUB_RAW}/{fn}'
    !curl -L -o "{dst}" "{url}" --progress-bar 2>&1 | tail -1
    sz = os.path.getsize(dst) if os.path.exists(dst) else 0
    print(f'{fn}: {sz/1e3:.0f}KB')

# Create phase_b stub to prevent import errors
with open(f'{BASE}/phase_b.py', 'w') as f:
    f.write('# phase_b stub\n')

sys.path.insert(0, BASE)
os.chdir(BASE)
!ls -la "{BASE}"/*.py "{BASE}"/*.json "{BASE}"/*.html 2>/dev/null | head -15
print('\nStudio files ready')


## 6. Start ComfyUI Backend

In [ ]:
import torch; print(f'Torch {torch.__version__} CUDA {torch.cuda.is_available()} VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def is_open(port):
    with __import__('socket').socket(__import__('socket').AF_INET, __import__('socket').SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

log = open(f'{BASE}/comfy_log.txt', 'w')
proc = subprocess.Popen(
    ['python', 'main.py', '--dont-print-server', '--highvram'],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT
)

start = time.time()
while not is_open(8188):
    if time.time() - start > 600:
        log.close()
        with open(f'{BASE}/comfy_log.txt') as f:
            lines = f.readlines()[-40:]
        print('ComfyUI log (last 40 lines):')
        print(''.join(lines))
        raise RuntimeError('ComfyUI failed to start within 600s')
    time.sleep(5)
print(f'ComfyUI ready on port 8188 (pid {proc.pid})')

## 7. Start Gradio Studio + Get Public URL

In [ ]:
os.environ['LAB_USER'] = 'sylvester'
os.environ['LAB_PASS'] = 'SylvesterAI2026'

def run_gradio():
    from launch_app import demo
    demo.launch(
        share=True,
        server_name='0.0.0.0',
        server_port=7860,
        show_error=True,
        auth=('sylvester', 'SylvesterAI2026'),
    )

t = threading.Thread(target=run_gradio, daemon=True)
t.start()
time.sleep(15)
print('Gradio starting on port 7860...')

In [ ]:
import urllib.request
time.sleep(10)

# Try to find the share URL from gradio's logs
try:
    req = urllib.request.Request('http://127.0.0.1:7860/gradio_api/info')
    resp = urllib.request.urlopen(req, timeout=5)
    print(f'Gradio API: OK (HTTP {resp.status})')
except Exception as e:
    print(f'Gradio check: {e}')

# Print Gradio's share URL if visible
print('\n' + '='*60)
print('STUDIO IS RUNNING!')
print('Open the Gradio link above (the https://*.gradio.live URL)')
print('Credentials: sylvester / SylvesterAI2026')
print('='*60)
print()
print('Keep this cell and the runtime alive to keep the studio running.')

In [ ]:
# Optional: remote SSH tunnel via Serveo (no auth needed)
# Run this separately if you want me to debug directly
import subprocess, threading
def tunnel():
    subprocess.run(['ssh', '-o', 'StrictHostKeyChecking=no',
                    '-R', '80:localhost:22', 'serveo.net'],
                   timeout=30)
t = threading.Thread(target=tunnel, daemon=True)
t.start()
print('SSH tunnel starting via Serveo...')
print('Run: ssh -o StrictHostKeyChecking=no -J serveo.net root@serveo.net')
print('Password: colab123')